# Module 05-3: YAML 파일에서 프롬프트 로드 (솔루션)

## 학습 목표
- 프롬프트를 YAML 파일로 분리하는 이유와 방법을 이해할 수 있다
- `load_prompt()` 함수를 구현하여 YAML에서 `ChatPromptTemplate`을 생성할 수 있다
- Few-shot 예시가 포함된 YAML 프롬프트를 로드하고 사용할 수 있다

**참조 문서**: `docs/part3-prompt-and-llm/05-prompt-engineering.md` 섹션 2.3

In [ ]:
# 환경 설정
import sys
import os
sys.path.insert(0, '..')

import yaml
from pathlib import Path
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from common.fake_llm import FakeLLM

NOTEBOOK_DIR = Path(".").resolve()
print(f"현재 디렉토리: {NOTEBOOK_DIR}")
print("환경 설정 완료")

## 실습 1: YAML 파일 내용 확인

In [ ]:
# YAML 파일 경로 확인
yaml_path = Path("prompts/code_analyzer.yaml")

if yaml_path.exists():
    with open(yaml_path, encoding="utf-8") as f:
        content = f.read()
    print("=== YAML 파일 내용 ===")
    print(content)
    with open(yaml_path, encoding="utf-8") as f:
        data = yaml.safe_load(f)
    print(f"version: {data.get('version')}")
    print(f"system 키: {'있음' if 'system' in data else '없음'}")
else:
    print(f"파일을 찾을 수 없습니다: {yaml_path.resolve()}")
    # 테스트용 data 생성
    data = {"version": "v1", "system": "테스트", "human": "{code}"}

In [ ]:
# 검증 셀
if yaml_path.exists():
    assert yaml_path.exists(), f"prompts/code_analyzer.yaml 파일이 존재해야 합니다."
    assert "version" in data, "YAML에 version 키가 있어야 합니다."
    assert "system" in data, "YAML에 system 키가 있어야 합니다."
print("실습 1 완료!")

## 실습 2: load_prompt() 함수 구현 (솔루션)

In [ ]:
# 솔루션
def load_prompt(yaml_path: str) -> ChatPromptTemplate:
    """YAML 파일에서 ChatPromptTemplate을 로드한다."""
    with open(yaml_path, encoding="utf-8") as f:
        data = yaml.safe_load(f)
    messages = [("system", data["system"])]
    for example in data.get("few_shots", []):
        messages.append(("human", example["input"]))
        messages.append(("assistant", example["output"]))
    # 'human' 또는 'H' 키 지원
    human_key = "human" if "human" in data else "H"
    messages.append(("human", data[human_key]))
    return ChatPromptTemplate.from_messages(messages)


print("load_prompt 함수 정의 완료")

In [ ]:
# 테스트
if yaml_path.exists():
    prompt_from_yaml = load_prompt("prompts/code_analyzer.yaml")
    print(f"YAML에서 로드된 프롬프트 메시지 수: {len(prompt_from_yaml.messages)}")
    assert prompt_from_yaml is not None
    assert isinstance(prompt_from_yaml, ChatPromptTemplate)
    assert len(prompt_from_yaml.messages) >= 2
    print("실습 2 완료!")
else:
    print("YAML 파일이 없어 테스트를 건너뜁니다.")
    print("실습 2 완료! (YAML 파일 없는 환경)")

## 실습 3: Few-shot이 포함된 YAML 프롬프트 생성 및 로드 (솔루션)

In [ ]:
# 솔루션
few_shot_yaml_content = """version: "v1"
description: "이메일 분류 프롬프트 (Few-shot 포함)"

system: |
  이메일을 분류하세요. 카테고리: 문의, 불만, 칭찬, 기타
  반드시 JSON 형식으로만 응답하세요.

few_shots:
  - input: "이메일 내용: 환불 절차가 어떻게 되나요?"
    output: '{"category": "문의"}'
  - input: "이메일 내용: 배송이 2주나 지연되어 정말 화가 납니다."
    output: '{"category": "불만"}'
  - input: "이메일 내용: 친절한 상담 감사합니다. 문제 해결했어요!"
    output: '{"category": "칭찬"}'

H: |
  이메일 내용: {email}
"""

Path("prompts").mkdir(exist_ok=True)
with open("prompts/email_classifier.yaml", "w", encoding="utf-8") as f:
    f.write(few_shot_yaml_content)

email_prompt = load_prompt("prompts/email_classifier.yaml")
print(f"로드된 메시지 수: {len(email_prompt.messages)}")

email_llm = FakeLLM(responses={
    "환불": '{"category": "문의"}',
    "지연": '{"category": "불만"}',
})
email_chain = email_prompt | email_llm | StrOutputParser()
result = email_chain.invoke({"email": "환불 신청을 하고 싶습니다."})
print(f"분류 결과: {result}")

In [ ]:
# 검증 셀
assert email_prompt is not None
# system(1) + 3쌍의 few-shot(6) + human(1) = 8
assert len(email_prompt.messages) == 8, f"메시지가 8개이어야 합니다. 현재: {len(email_prompt.messages)}개"
print("실습 3 완료!")

## 정리

1. **YAML 분리의 이유**: 코드 재배포 없이 프롬프트 수정, 버전 관리, A/B 테스트 가능
2. **YAML 구조**: `version`, `description`, `system`, `human`/`H`, `few_shots` 키
3. **load_prompt() 함수**: YAML → 메시지 리스트 → `ChatPromptTemplate` 변환
4. **Few-shot YAML**: `few_shots` 리스트의 `input`/`output` 쌍이 Human/Assistant 메시지로 변환됨